In [ ]:
import random, time
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import timm


In [ ]:
# Mode
TRAINING_MODE = True   # set False at submission

# Audio
SR         = 32000
DURATION   = 5
N_MELS     = 128
HOP_LENGTH = 512
N_FFT      = 1024
FMIN       = 50
FMAX       = 14000

# Training
BATCH_SIZE = 64
EPOCHS     = 8
LR         = 1e-3
PATIENCE   = 4
SEED       = 42

# Paths
BASE_DIR    = Path('/kaggle/input/competitions/birdclef-2026')
TRAIN_AUDIO = BASE_DIR / 'train_audio'
TRAIN_SC    = BASE_DIR / 'train_soundscapes'
TEST_SC     = BASE_DIR / 'test_soundscapes'
META_CSV    = BASE_DIR / 'train.csv'
TAX_CSV     = BASE_DIR / 'taxonomy.csv'
SC_CSV      = BASE_DIR / 'train_soundscapes_labels.csv'
SAMPLE_SUB  = BASE_DIR / 'sample_submission.csv'

SPEC_DIR    = Path('/kaggle/working/spectrograms')
SC_SPEC_DIR = Path('/kaggle/working/sc_spectrograms')
MODEL_PATH = Path('/kaggle/working/best_model.pt')
if not TRAINING_MODE:
    MODEL_PATH = Path('/kaggle/input/birdclef2026-model/best_model.pt')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f'Device : {DEVICE}')
print(f'Mode   : {"TRAINING" if TRAINING_MODE else "SUBMISSION"}')


In [ ]:
df  = pd.read_csv(META_CSV)
tax = pd.read_csv(TAX_CSV)
sc  = pd.read_csv(SC_CSV)

NUM_CLASSES    = len(tax)
species_list   = tax['primary_label'].astype(str).tolist()
species_to_idx = {sp: i for i, sp in enumerate(species_list)}

# Drop species with <2 clips so stratified split never crashes
counts   = df['primary_label'].value_counts()
valid_df = df[df['primary_label'].isin(counts[counts >= 2].index)].reset_index(drop=True)

print(f'Species to predict : {NUM_CLASSES}')
print(f'Training clips     : {len(valid_df)}')
print(f'Soundscape windows : {len(sc)}')


## EDA

In [ ]:
if TRAINING_MODE:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    # 1. Long-tail class distribution
    counts_sorted = valid_df['primary_label'].value_counts().values
    axes[0].bar(range(len(counts_sorted)), counts_sorted, width=1.0, color='steelblue')
    axes[0].set_title('Recordings per species (long tail)')
    axes[0].set_xlabel('Species rank')
    axes[0].set_ylabel('Count')

    # 2. Species coverage gap
    train_species = set(valid_df['primary_label'].astype(str))
    tax_species   = set(tax['primary_label'].astype(str))
    n_missing     = len(tax_species - train_species)
    axes[1].bar(['Train audio (206)', 'Zero-shot (28)'],
                [len(train_species), n_missing],
                color=['steelblue', 'salmon'])
    axes[1].set_title('Species coverage: 206 with audio, 28 zero-shot')
    axes[1].set_ylabel('Count')

    # 3. Domain gap: clean training clip spectrogram
    clean_row      = valid_df[valid_df['class_name'] == 'Aves'].iloc[0]
    audio_clean, _ = librosa.load(str(TRAIN_AUDIO / clean_row['filename']),
                                   sr=SR, duration=DURATION)
    mel_clean = librosa.power_to_db(
        librosa.feature.melspectrogram(y=audio_clean, sr=SR, n_mels=N_MELS,
                                       hop_length=HOP_LENGTH, n_fft=N_FFT,
                                       fmin=FMIN, fmax=FMAX), ref=np.max)
    axes[2].imshow(mel_clean, aspect='auto', origin='lower', cmap='viridis')
    axes[2].set_title('Training clip mel-spectrogram (clean, single species)')
    axes[2].set_xlabel('Time frames')
    axes[2].set_ylabel('Mel bins')

    plt.tight_layout()
    plt.show()

    print(valid_df['class_name'].value_counts().to_string())


## Audio Utilities

In [ ]:
def load_audio(path, start_sec=0.0):
    audio, _ = librosa.load(str(path), sr=SR, offset=start_sec, duration=DURATION)
    target = SR * DURATION
    if len(audio) < target:
        audio = np.pad(audio, (0, target - len(audio)))
    return audio[:target]

def middle_offset(filepath):
    total = librosa.get_duration(path=str(filepath))
    if total <= DURATION:
        return 0.0
    return (total - DURATION) / 2.0

def audio_to_melspec(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_mels=N_MELS,
        hop_length=HOP_LENGTH, n_fft=N_FFT,
        fmin=FMIN, fmax=FMAX,
    )
    mel = librosa.power_to_db(mel, ref=np.max)
    lo, hi = mel.min(), mel.max()
    if hi > lo:
        mel = (mel - lo) / (hi - lo)
    return mel.astype(np.float32)


In [ ]:
def apply_specaug(spec):
    spec = spec.copy()
    _, H, W = spec.shape
    for _ in range(2):           # 2 frequency masks
        f  = random.randint(0, 20)
        f0 = random.randint(0, max(0, H - f))
        spec[:, f0:f0 + f, :] = 0.0
    for _ in range(2):           # 2 time masks
        t  = random.randint(0, 40)
        t0 = random.randint(0, max(0, W - t))
        spec[:, :, t0:t0 + t] = 0.0
    return spec


## Preprocessing

In [ ]:
if TRAINING_MODE:
    SPEC_DIR.mkdir(parents=True, exist_ok=True)
    SC_SPEC_DIR.mkdir(parents=True, exist_ok=True)

    # Training clips
    print(f'Preprocessing {len(valid_df)} training clips...')
    saved = skipped = 0
    for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc='Clips'):
        npy = SPEC_DIR / (row['filename'].replace('/', '_').replace('.ogg', '.npy'))
        if npy.exists():
            skipped += 1
            continue
        fp    = TRAIN_AUDIO / row['filename']
        audio = load_audio(fp, start_sec=middle_offset(fp))
        if np.abs(audio).max() < 1e-4:
            skipped += 1
            continue
        np.save(npy, audio_to_melspec(audio))
        saved += 1
    print(f'Clips      -- saved: {saved:>5}  skipped: {skipped:>5}')

    # Soundscape windows
    print(f'Preprocessing {len(sc)} soundscape windows...')
    saved = skipped = 0
    for _, row in tqdm(sc.iterrows(), total=len(sc), desc='Soundscape windows'):
        parts     = str(row['start']).split(':')
        start_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        npy = SC_SPEC_DIR / f"{Path(row['filename']).stem}_{start_sec:06d}.npy"
        if npy.exists():
            skipped += 1
            continue
        audio = load_audio(TRAIN_SC / row['filename'], start_sec=float(start_sec))
        np.save(npy, audio_to_melspec(audio))
        saved += 1
    print(f'SC windows -- saved: {saved:>5}  skipped: {skipped:>5}')


## Datasets

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df       = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        npy = SPEC_DIR / (row['filename'].replace('/', '_').replace('.ogg', '.npy'))

        if npy.exists():
            spec = np.load(npy)
        else:
            fp   = TRAIN_AUDIO / row['filename']
            spec = audio_to_melspec(load_audio(fp, start_sec=middle_offset(fp)))

        spec = np.stack([spec, spec, spec])   # (3, H, W)
        if self.is_train:
            spec = apply_specaug(spec)

        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        sp = str(row['primary_label'])
        if sp in species_to_idx:
            target[species_to_idx[sp]] = 1.0

        return torch.tensor(spec, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)


In [ ]:
class SoundscapeWindowDataset(Dataset):
    def __init__(self, sc_df, is_train=False):
        self.is_train = is_train
        self.samples  = []
        for _, row in sc_df.iterrows():
            parts     = str(row['start']).split(':')
            start_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
            self.samples.append({
                'filepath' : TRAIN_SC / row['filename'],
                'start_sec': float(start_sec),
                'npy'      : SC_SPEC_DIR / f"{Path(row['filename']).stem}_{start_sec:06d}.npy",
                'labels'   : [s.strip() for s in str(row['primary_label']).split(';')],
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        if s['npy'].exists():
            spec = np.load(s['npy'])
        else:
            spec = audio_to_melspec(load_audio(s['filepath'], start_sec=s['start_sec']))

        spec = np.stack([spec, spec, spec])
        if self.is_train:
            spec = apply_specaug(spec)

        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        for sp in s['labels']:
            if sp in species_to_idx:
                target[species_to_idx[sp]] = 1.0

        return torch.tensor(spec, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)


## Data Loaders

In [ ]:
def seed_worker(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)

if TRAINING_MODE:
    g = torch.Generator()
    g.manual_seed(SEED)

    # Clip train split (stratified by species, 90/10)
    train_clips, _ = train_test_split(
        valid_df, test_size=0.1,
        stratify=valid_df['primary_label'],
        random_state=SEED,
    )

    # Soundscape file-level 80/20 split -- val files never seen during training
    val_files = pd.Series(sc['filename'].unique()).sample(frac=0.2, random_state=SEED)
    sc_val   = sc[sc['filename'].isin(val_files)].reset_index(drop=True)
    sc_train = sc[~sc['filename'].isin(val_files)].reset_index(drop=True)

    train_ds = ConcatDataset([
        BirdDataset(train_clips, is_train=True),
        SoundscapeWindowDataset(sc_train, is_train=True),
    ])
    val_ds = SoundscapeWindowDataset(sc_val, is_train=False)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=g,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True,
    )

    print(f'Train samples : {len(train_ds)}')
    print(f'Val samples   : {len(val_ds)}')
    print(f'Train batches : {len(train_loader)}')


## Model

In [ ]:
class BirdClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=TRAINING_MODE,
            in_chans=3, num_classes=NUM_CLASSES,
        )

    def forward(self, x):
        return self.backbone(x)


## Training

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc='  train', leave=False)
    for specs, targets in pbar:
        specs, targets = specs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
            loss = criterion(model(specs), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    return total_loss / len(loader)


def eval_on_soundscapes(model, loader):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for specs, targets in tqdm(loader, desc='  eval', leave=False):
            probs = torch.sigmoid(model(specs.to(DEVICE))).cpu().numpy()
            all_probs.append(probs)
            all_targets.append(targets.numpy())
    probs   = np.concatenate(all_probs)
    targets = np.concatenate(all_targets)
    aucs = [
        roc_auc_score(targets[:, i], probs[:, i])
        for i in range(NUM_CLASSES)
        if 0 < targets[:, i].sum() < len(targets)
    ]
    return float(np.mean(aucs))


### Training Loop

In [ ]:
if TRAINING_MODE:
    model      = BirdClassifier().to(DEVICE)
    criterion  = nn.BCEWithLogitsLoss()
    optimizer  = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler     = torch.amp.GradScaler(DEVICE, enabled=(DEVICE == 'cuda'))

    best_auc       = 0.0
    patience_count = 0

    for epoch in range(1, EPOCHS + 1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_auc    = eval_on_soundscapes(model, val_loader)
        scheduler.step()
        elapsed    = time.time() - t0

        print(f'Epoch {epoch}/{EPOCHS}  loss {train_loss:.4f}  val AUC {val_auc:.4f}  ({elapsed:.0f}s)')

        if val_auc > best_auc:
            best_auc       = val_auc
            patience_count = 0
            torch.save(model.state_dict(), MODEL_PATH)
            print(f'  Saved best model (AUC {best_auc:.4f})')
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    print(f"Training complete. Best val AUC: {best_auc:.4f}")

## Inference & Submission

In [ ]:
def predict_soundscape(filepath, model):
    audio, _ = librosa.load(str(filepath), sr=SR)
    n_samples = len(audio)
    chunk     = SR * DURATION
    rows      = []
    stem      = Path(filepath).stem

    model.eval()
    with torch.no_grad():
        offset = 0
        while offset + chunk <= n_samples:
            spec    = audio_to_melspec(audio[offset:offset + chunk])
            spec    = np.stack([spec, spec, spec])
            tensor  = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            probs   = torch.sigmoid(model(tensor)).cpu().numpy()[0]
            end_sec = (offset + chunk) // SR
            rows.append((f'{stem}_{end_sec}', probs))
            offset += chunk

    return rows


In [ ]:
# Load model
print(f'Loading model from {MODEL_PATH}')
infer_model = BirdClassifier().to(DEVICE)
infer_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
infer_model.eval()
print('Model loaded.')


In [ ]:
# Run inference on all test soundscapes
test_files = sorted(TEST_SC.glob('*.ogg'))
print(f'Found {len(test_files)} test files')

all_rows = []
for fp in tqdm(test_files, desc='Test soundscapes'):
    all_rows.extend(predict_soundscape(fp, infer_model))

print(f'Predicted {len(all_rows)} windows from {len(test_files)} files')


In [ ]:
# Build submission aligned to sample_submission column order
sample_sub = pd.read_csv(SAMPLE_SUB)
col_order  = list(sample_sub.columns)

if all_rows:
    rows_data = [
        dict([('row_id', row_id)] + list(zip(species_list, probs)))
        for row_id, probs in all_rows
    ]
    sub_df = pd.DataFrame(rows_data)
    sub_df = sample_sub[['row_id']].merge(sub_df, on='row_id', how='left').fillna(0.5)
else:
    sub_df = sample_sub[['row_id']].copy()
    for col in species_list:
        sub_df[col] = 0.5

sub_df = sub_df[col_order]
sub_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'submission.csv saved -- {sub_df.shape[0]} rows x {sub_df.shape[1]} cols')
